# BDT Training (XGBoost)

<div style="text-align: justify">

The following notebook is dedicated to supervised machine learning training for the <b>Tau Supersymmetry</b> search analysis. A gradient-boosted decision tree classifier is trained using <b>XGBoost</b> on the rectangular MC DataFrame produced by the feature engineering pipeline. Two split strategies are supported: a single stratified train/test split and stratified K-fold cross-validation.

</div>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis and model configuration |
| Load | `io.load_dataframe` | Read mc.parquet from feature engineering output |
| Labels | — | Derive ordered class names from `eventOrigin` |
| Prepare | `splits.prepare_features_target` | Separate training features, labels, and weights |
| Split | `splits.train_test_split` / `splits.kfold_split` | Stratified train/test or K-fold split |
| Params | `bdt.build_params` | Build XGBoost kwargs from Hydra config |
| Train | `bdt.train` / `bdt.train_kfold` | Fit model(s) with early stopping |
| Curves | `models.plots.plot_training_curves` | Visualise convergence |
| Predict | `bdt.predict` | Compute hard predictions and probability scores |
| Save | `bdt.save_model`, `io.save_dataframe` | Persist model and predictions parquet |

The same pipeline is available as a CLI via `python train.py` or `make train`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data Processing:
* [Pandas](https://pandas.pydata.org/)
* [NumPy](https://numpy.org/)

Data Visualization:
* [Matplotlib](https://matplotlib.org/)

Machine Learning:
* [XGBoost](https://xgboost.readthedocs.io/en/stable/)
* [scikit-learn](https://scikit-learn.org/stable/)

Serialization:
* [Apache Parquet](https://parquet.apache.org/)

### Notebook

Activating autoreload of imported modules.

In [ ]:
%load_ext autoreload
%autoreload 2

Initializing the project root.

In [ ]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [ ]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration.  All analysis parameters (run, region, channel, model hyperparameters, split strategy) are defined in `configs/` and can be overridden here.

> **Device note:** the model config defaults to `device: cuda`. Change it to `device: cpu` for local CPU execution.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config")

In [ ]:
cfg

Resolving input and output directories from config.

In [ ]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
models_dir = path / output_paths["models_dir"]
plots_dir = path / output_paths["plots_dir"] / "bdt"

models_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

## Deserialization

Loading the MC DataFrame produced by the feature engineering pipeline.

In [ ]:
from src.processing.io import load_dataframe

df_mc = load_dataframe(dataframes_dir / "mc.parquet")
print(f"Loaded: {len(df_mc):,} events, {len(df_mc.columns)} columns")

Deriving ordered class names from `eventOrigin` for use in outputs and plots.

In [ ]:
class_names = df_mc.groupby("class")["eventOrigin"].first().sort_index().tolist()
n_classes = len(class_names)
print(f"Classes ({n_classes}): {class_names}")

## Training Preparation

### Features & Target

Separating training features, integer class labels, and per-event sample weights. Non-training metadata columns (`class`, `class_weight`, `tau_n`, `eventOrigin`) are excluded from the feature matrix.

In [ ]:
from src.models.splits import prepare_features_target

X, y, weights = prepare_features_target(df_mc)
print(f"Feature matrix: {X.shape}")
print(f"Class distribution:\n{y.value_counts().sort_index().rename(index=dict(enumerate(class_names)))}")

### Train / Test Split

Splitting the dataset according to the strategy defined in `configs/pipeline/default.yaml` (`split_strategy`). Both strategies use stratified sampling to preserve the class distribution.

* **`train_test`** — single stratified split with `test_split` fraction held out.
* **`k_fold`** — stratified K-fold; each fold's test set is non-overlapping, giving out-of-fold (OOF) coverage of the full dataset.

In [ ]:
from src.models.splits import train_test_split, kfold_split

split_strategy = cfg.pipeline.split_strategy

if split_strategy == "train_test":
    X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
        X, y, weights,
        test_size=cfg.data.test_split,
        seed=cfg.seed,
    )
    print(f"Train: {len(X_train):,} events | Test: {len(X_test):,} events")

elif split_strategy == "k_fold":
    folds = kfold_split(X, y, weights, n_splits=cfg.pipeline.n_splits, seed=cfg.seed)
    print(f"K-fold: {cfg.pipeline.n_splits} stratified folds")

else:
    raise ValueError(f"Unknown split_strategy: {split_strategy!r}")

## Training

Building XGBoost constructor parameters from the Hydra model config. The objective and evaluation metric are set automatically based on the number of classes.

In [ ]:
from src.models.bdt import build_params

params = build_params(cfg, n_classes=n_classes)
params

Training the model with early stopping. Both the training and validation sets are monitored; XGBoost stops when the validation metric does not improve for `early_stopping_rounds` consecutive rounds.

In [ ]:
from src.models.bdt import train, train_kfold

if split_strategy == "train_test":
    model = train(
        params,
        X_train, y_train,
        X_test, y_test,
        w_train=w_train,
        early_stopping_rounds=cfg.pipeline.early_stopping_rounds,
    )
    print(f"Best iteration : {model.best_iteration}")
    print(f"Best val score : {model.best_score:.6f}")

elif split_strategy == "k_fold":
    models, y_pred, y_proba, y_test = train_kfold(
        params,
        folds,
        early_stopping_rounds=cfg.pipeline.early_stopping_rounds,
    )
    print(f"Trained {len(models)} folds")
    best_iters = [m.best_iteration for m in models]
    print(f"Best iterations per fold: {best_iters}")

## Training Curves

Plotting per-round training and validation loss.  The vertical dashed line marks the best iteration selected by early stopping.

In [ ]:
from src.models.bdt import get_evals_result
from src.models.plots import plot_training_curves, plot_kfold_training_curves
from src.visualization.plots import save_figure

metric = params["eval_metric"]

if split_strategy == "train_test":
    fig = plot_training_curves(get_evals_result(model), metric=metric)
    save_figure(fig, plots_dir / "training_curves.png")
    fig.show()

elif split_strategy == "k_fold":
    fig = plot_kfold_training_curves(models, metric=metric)
    save_figure(fig, plots_dir / "training_curves_kfold.png")
    fig.show()

## Predictions

Computing hard predictions and per-class probability scores on the test set.

In [ ]:
from src.models.bdt import predict, build_predictions_frame

if split_strategy == "train_test":
    y_pred, y_proba = predict(model, X_test)

# For k_fold, y_pred / y_proba / y_test are already set by train_kfold above

predictions_df = build_predictions_frame(y_test, y_pred, y_proba, class_names)
print(f"Predictions DataFrame: {predictions_df.shape}")
predictions_df.head()

## Serialization

Saving the trained model and the test-set predictions.  The predictions parquet is consumed by the downstream evaluation step.

In [ ]:
from src.models.bdt import save_model
from src.processing.io import save_dataframe

if cfg.pipeline.save_model:
    if split_strategy == "train_test":
        save_model(model, models_dir / "bdt.ubj")
    elif split_strategy == "k_fold":
        for fold_idx, m in enumerate(models):
            save_model(m, models_dir / f"bdt_fold{fold_idx}.ubj")

save_dataframe(predictions_df, dataframes_dir / "bdt_predictions.parquet")